# Quality benchmark — analysis

The generated pages hold the facts: [how it is run](../docs/benchmarks/quality-docs.md) and
[what came out](../docs/benchmarks/quality-results.md). This notebook is where the judgement
goes.

The benchmark's question is **model against expert annotation**, so the model is the primary
index everywhere below and the dataset sits inside it. Model-against-model numbers are here to
raise suspicions about the annotation, never to crown anything.

Run it after `python -m benchmarks --benchmark quality`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path('..').resolve() / 'results' / 'quality'
STORE = Path('..').resolve() / '.atlas_data'
WORTH = ['good', 'usable']

def per_image() -> pd.DataFrame:
    """Every model's answer about every photograph, one row each."""
    frames = []
    for path in sorted(RESULTS.glob('*/*.csv')):
        frame = pd.read_csv(path)
        frame['model'] = path.parent.name
        frame['dataset'] = path.stem
        frames.append(frame)
    answers = pd.concat(frames, ignore_index=True)
    answers['worth_measuring'] = answers['grade'].isin(WORTH)
    # A model that names a grade is taken at its word; one that publishes only a confidence
    # is read at its own published threshold, which for the toolbox ensemble is 0.5.
    answers['kept'] = np.where(
        answers['verdict'].notna() & (answers['verdict'] != ''),
        answers['verdict'].isin(WORTH),
        answers['gradeable'] >= 0.5,
    )
    answers['right'] = answers['worth_measuring'] == answers['kept']
    return answers

def summaries() -> pd.DataFrame:
    rows = []
    for path in sorted(RESULTS.glob('*/*.json')):
        record = json.loads(path.read_text())
        s = record['summary']
        rows.append({
            'model': record['model'], 'dataset': record['dataset'],
            'processed': s['processed'], 'total': s['total'], 'complete': s['complete'],
            'coverage': s['coverage'],
            'accuracy': s['gradeable']['accuracy'], 'kappa': s['gradeable']['kappa'],
            'roc_auc': s['gradeable']['roc_auc'],
        })
    return pd.DataFrame(rows).set_index(['model', 'dataset'])

answers = per_image()
graded = answers[answers['outcome'] == 'graded']
summaries()



## 1. Coverage first

A model that declines a photograph has not got it wrong. Coverage is the share of a dataset a
model was willing to answer for at all, and it is read beside accuracy rather than folded into
it.


In [ ]:
answers.pivot_table(index=['model', 'dataset'], columns='outcome', values='key',
                    aggfunc='count').fillna(0).astype(int)


## 2. Against the expert's grade

The benchmark's primary question, in the form that answers it honestly: **what the dataset's
readers called worth measuring, against what the model called worth measuring**. The two
errors are not interchangeable — discarding a good photograph costs a study its sample size,
keeping a bad one costs it its measurements — and a single accuracy hides which of the two a
model commits.

A model that names a grade is taken at its word. The
[toolbox ensemble](../docs/models/fit-quality.md) names none: it publishes a confidence and
calls everything **at or above 0.5** gradeable and everything below it ungradeable. That is
its own default, and its authors say plainly the threshold does not transfer between datasets.


In [ ]:
def confusion(frame: pd.DataFrame) -> pd.Series:
    """The two-by-two table, with the numbers that summarise it.

    Accuracy flatters a model on a dataset where one class dominates: a grader that keeps
    everything scores 0.9 on a collection that is 90% gradeable while agreeing with nobody about
    anything. Cohen's κ measures agreement beyond what guessing the common answer achieves, so the
    two together say what neither says alone. κ is undefined where the reference has one class.
    """
    kept_worth = (frame['worth_measuring'] & frame['kept']).sum()
    lost_worth = (frame['worth_measuring'] & ~frame['kept']).sum()
    kept_bad = (~frame['worth_measuring'] & frame['kept']).sum()
    lost_bad = (~frame['worth_measuring'] & ~frame['kept']).sum()
    worth, bad = kept_worth + lost_worth, kept_bad + lost_bad
    total = len(frame)
    agreed = (kept_worth + lost_bad) / total if total else np.nan
    # κ is undefined where the reference has only one class: there is no chance agreement to take
    # out, and the formula collapses to zero, which would read as disagreement rather than as the
    # absence of a question.
    by_chance = (
        (worth * (kept_worth + kept_bad) + bad * (lost_worth + lost_bad)) / total**2
        if total and worth and bad else np.nan
    )
    return pd.Series({
        'photographs': total,
        'kept, worth measuring': kept_worth,
        'discarded, worth measuring': lost_worth,
        'kept, not worth measuring': kept_bad,
        'discarded, not worth measuring': lost_bad,
        'share of good kept': kept_worth / worth if worth else np.nan,
        'share of bad discarded': lost_bad / bad if bad else np.nan,
        'accuracy': agreed,
        'kappa': (agreed - by_chance) / (1 - by_chance) if by_chance < 1 else np.nan,
    })

tabulated = graded.groupby(['model', 'dataset']).apply(confusion, include_groups=False)
tabulated.round(3)



### 2.1 The same table, drawn

Twenty rows of counts are not read at a glance; a grid of two-by-two tables is. Models run
down and datasets across, and each cell holds the number of photographs. A model whose weight
sits in the top-right corner throws good photographs away; one whose weight sits in the
bottom-left keeps unusable ones.


In [ ]:
datasets = sorted(graded['dataset'].unique())
figure, axes = plt.subplots(
    len(models), len(datasets), figsize=(2.3 * len(datasets), 2.5 * len(models)), squeeze=False
)
for down, model in enumerate(models):
    for across, dataset in enumerate(datasets):
        axis = axes[down][across]
        frame = graded[(graded['model'] == model) & (graded['dataset'] == dataset)]
        counts = np.array([
            [(frame['worth_measuring'] & frame['kept']).sum(),
             (frame['worth_measuring'] & ~frame['kept']).sum()],
            [(~frame['worth_measuring'] & frame['kept']).sum(),
             (~frame['worth_measuring'] & ~frame['kept']).sum()],
        ])
        shares = counts / counts.sum(axis=1, keepdims=True).clip(min=1)
        axis.imshow(shares, cmap='Blues', vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                axis.text(j, i, f'{counts[i, j]}', ha='center', va='center', fontsize=8,
                          color='white' if shares[i, j] > 0.5 else 'black')
        axis.set_xticks([0, 1], ['model\nkeeps', 'model\ndiscards'], fontsize=6)
        axis.set_yticks([0, 1], ['expert:\nworth it', 'expert:\nnot'], fontsize=6)
        if down == 0:
            axis.set_title(dataset, fontsize=9)
        if across == 0:
            axis.set_ylabel(model, fontsize=7)
figure.suptitle('what the readers said (rows) against what the model said (columns)', fontsize=9)
figure.tight_layout()


### 2.2 Accuracy and agreement beyond chance

Accuracy flatters a model on a dataset where one class dominates — a grader that keeps
everything scores well on a collection that is mostly gradeable while agreeing with nobody
about anything. Cohen's κ measures what is left after chance agreement is taken out. Where the
two disagree, that gap is the finding.

κ is undefined on a dataset whose reference has only one class, so PAPILA has no bar.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for axis, metric in zip(axes, ['accuracy', 'kappa']):
    tabulated[metric].unstack('dataset').plot.bar(ax=axis, width=0.8)
    axis.set_title(metric)
    axis.set_xlabel('')
    axis.axhline(0, color='black', linewidth=0.6)
    axis.tick_params(axis='x', labelrotation=20, labelsize=7)
    axis.legend(fontsize=6, title=None)
figure.tight_layout()


### 2.3 The whole ROC curve, not one point on it

Accuracy depends on where a model's threshold sits; the area under the curve does not. Where
two curves cross, the better model depends on whether you would rather miss a bad photograph
or throw away a good one — the reader's decision, not ours.

Datasets whose reference contains no bad photographs are left out: a curve needs both classes.
They appear in section 6.



In [ ]:
def curve(frame):
    worth = frame['worth_measuring'].to_numpy()[np.argsort(-frame['gradeable'].to_numpy())]
    return (np.r_[0, np.cumsum(~worth) / max((~worth).sum(), 1)],
            np.r_[0, np.cumsum(worth) / max(worth.sum(), 1)])

models = sorted(graded['model'].unique())
both = sorted(d for d, f in graded.groupby('dataset') if f['worth_measuring'].nunique() == 2)
print('no ranking is defined on:', sorted(set(graded['dataset']) - set(both)))

figure, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 3.6), squeeze=False)
for axis, model in zip(axes[0], models):
    for dataset in both:
        frame = graded[(graded['model'] == model) & (graded['dataset'] == dataset)]
        if len(frame):
            axis.plot(*curve(frame), label=dataset, linewidth=1)
    axis.plot([0, 1], [0, 1], color='grey', linewidth=0.5)
    axis.set_title(model, fontsize=8)
    axis.set_xlabel('kept, though bad')
    axis.set_ylabel('kept, and worth measuring')
    axis.legend(fontsize=6)
figure.tight_layout()


### 2.4 FIVES: which defect is a model actually reacting to?

[FIVES](../docs/datasets/fives.md) publishes no overall grade. It publishes three binary
component scores — illumination and contrast, blur, low contrast — and this repository derives
the grade from them: all three sound is `good`, one short is `usable`, worse is `bad`.

That makes it the one dataset here that can say **which defect a model notices**. A model that
reacts to blur and ignores illumination is a different instrument from one that does the
reverse, and no summary number tells them apart. The photograph count is printed beside each
rate because the rare combinations hold one or two images and their rates are noise.



In [ ]:
manifest = pd.read_csv(STORE / 'fives' / 'manifest.csv')
components = ['illumination_contrast', 'blur', 'low_contrast']
fives = graded[graded['dataset'] == 'fives'].merge(
    manifest[['key', *components]], on='key', how='left'
)

if len(fives):
    kept = fives.pivot_table(index=components, columns='model', values='kept', aggfunc='mean')
    kept['photographs'] = fives.groupby(components)['key'].nunique()
    display(kept.round(3).sort_values('photographs', ascending=False))


## 3. In-sample and out-of-sample

The same table as section 2, grouped by whether the model had seen these photographs while it
was learning. What each model trained on, from its catalogue page:

| Model | Trained on |
| --- | --- |
| [fit-quality](../docs/models/fit-quality.md) | DeepDRiD and DRIMDB — split not stated, so all of both is in-sample |
| [automorph-quality-grader](../docs/models/automorph-quality-grader.md) | EyeQ's training split |
| [quickqual](../docs/models/quickqual.md) | EyeQ's training split |
| [quickqual-meme](../docs/models/quickqual-meme.md) | EyeQ's training split, the same parameters |
| [vascx-quality](../docs/models/vascx-quality.md) | not published; its own checkpoint names EyeQ |

None of the datasets measured here is in any of those lists, so every row below is
out-of-sample — except VascX, which is `unknown` rather than cleared, because nobody published
what it trained on. Where a dataset's own splits carry different marks, the split breakdown is
the only table that can be read.


In [ ]:
import sys
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from benchmarks import contamination

marked = graded.copy()
marked['marked'] = [
    contamination.mark(model, dataset, split)
    for model, dataset, split in zip(marked['model'], marked['dataset'], marked['split'])
]
marked.groupby(['model', 'marked', 'dataset', 'split']).apply(
    confusion, include_groups=False
).round(3)


## 4. Where the models disagree with each other

Two models agreeing is not two pieces of evidence when they learned from the same labels — and
three of these were fitted on EyeQ. Agreement with a model trained on other data is the
informative number, and disagreement is where to go looking for a bad annotation.


In [ ]:
verdicts = graded.pivot_table(index=['dataset', 'key'], columns='model', values='kept')
pairs = list(verdicts.columns)
pd.DataFrame(
    [[(verdicts[a] == verdicts[b]).mean() for b in pairs] for a in pairs],
    index=pairs, columns=pairs,
).round(3)


## 5. Where the models disagree with the readers

Two of these datasets kept their readers apart, so the photographs the readers themselves
disagreed about can be separated from the ones they were sure of. A model that is wrong where
the humans disagreed is in different trouble from one that is wrong where they did not — and
the first kind is a reason to look again at the reference rather than at the software.


In [ ]:
def readers(entry) -> list[str]:
    return [part.split('=')[1] for part in str(entry).split(';') if '=' in part]

multi = graded[graded['readers'].fillna('') != ''].copy()
multi['unanimous'] = multi['readers'].map(lambda entry: len(set(readers(entry))) == 1)
multi.pivot_table(index=['model', 'dataset'], columns='unanimous', values='right').round(3)


## 6. What a model would throw away

[PAPILA](../docs/datasets/papila.md) publishes no quality grades. It is here on this
repository's assumption that all 488 of its photographs are sound — one camera, disc-centred,
every frame outlined by two ophthalmologists. That makes it useless for asking whether a model
finds bad photographs, and the only place here that answers a question every user of these
pipelines has: run this gate over a clean dataset, and how much of it disappears?

**The black bands are not the explanation.** PAPILA's retina fills the frame vertically, so the
square the store builds is nearly a fifth canvas — and a quality model judges the square it is
handed. Trimming those bands off and handing the model the photograph's own 4:3 rectangle has
come out the opposite way round from the obvious guess: the square is the familiar shape for
models fitted on screening photographs, so the canvas is not what makes them reject it.


In [ ]:
assumed = [d for d in graded['dataset'].unique() if d == 'papila']
if assumed:
    clean = graded[graded['dataset'].isin(assumed)].copy()
    # Empty for a model no pipeline gates on, and pandas reads what is left as booleans or
    # as numbers depending on what it is concatenated with, so accept both spellings.
    carried = clean['carried_by_its_pipeline'].astype(str).str.lower()
    clean['kept_by_its_pipeline'] = carried.map(
        {'true': 1.0, '1.0': 1.0, 'false': 0.0, '0.0': 0.0}
    )
    display(clean.groupby('model')[['kept', 'kept_by_its_pipeline']].mean().round(3))


## 7. The hard cases, dataset by dataset

Up to eight photographs from **each** dataset that every model got wrong. Taken per dataset
rather than from the pile: a grid drawn from whichever dataset fails most is a picture of that
dataset, not of the benchmark. Each is labelled with the dataset it came from and the grade it
was given.

A table says a model scores 0.82. This is where someone finds out whether the 0.82 is two
populations, whether one of them is a camera — or whether the grade itself is wrong.


In [ ]:
from PIL import Image

PER_DATASET = 8
wrong = graded.groupby(['dataset', 'key'])['right'].mean()

for dataset in sorted(graded['dataset'].unique()):
    hardest = wrong.loc[dataset][wrong.loc[dataset] == 0].index[:PER_DATASET]
    if not len(hardest):
        print(f'{dataset}: no photograph that every model got wrong')
        continue
    print(f'{dataset}: {len(hardest)} of {int((wrong.loc[dataset] == 0).sum())} '
          f'photographs every model got wrong')
    figure, axes = plt.subplots(1, len(hardest), figsize=(2.2 * len(hardest), 2.6))
    for axis, key in zip(np.atleast_1d(axes), hardest):
        axis.imshow(Image.open(STORE / dataset / '512' / 'images' / f'{key}.png'))
        said = graded[(graded['dataset'] == dataset) & (graded['key'] == key)]
        axis.set_title(f"{key}\n{dataset} says {said['grade'].iloc[0]}", fontsize=7)
        axis.axis('off')
    figure.tight_layout()
    plt.show()


## 8. What this benchmark cannot say

- **A model marked `unknown` is not cleared.** VascX publishes no list of what it trained on,
  and its shipped configuration names EyeQ for this model, so no out-of-sample claim can be
  made for it here.
- **The three references are not the same statement.** One dataset's grade is its own, one is
  derived by this repository from components, one is assumed outright — and one of them never
  uses the middle class at all.
- **An assumed reference measures what a model discards, not how accurate it is.**
- **Nothing here says a model is best.** Read a row, not a column.
